# United Kingdom YouTube Trending Videos

Profile notebook for `youtube_trending_united_kingdom_processed.csv`.


This notebook is generated from the preprocessing metadata so it stays aligned with the latest cleaned dataset and can be rerun after each pipeline refresh.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dataset").exists() and (candidate / "src").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PREPROCESS_DIR = PROJECT_ROOT / "dataset" / "processed_data"
DATASET_FILE = PREPROCESS_DIR / "youtube_trending_united_kingdom_processed.csv"
CATALOG_PATH = PREPROCESS_DIR / "dataset_catalog.json"

with CATALOG_PATH.open("r", encoding="utf-8") as handle:
    catalog = json.load(handle)

metadata = next(item for item in catalog["datasets"] if item["name"] == "youtube_trending_united_kingdom")
parse_dates = ['trending_date']

df = pd.read_csv(DATASET_FILE, parse_dates=parse_dates or None)
df.head()


In [ ]:
pd.DataFrame(
    {
        "Metric": [
            "Source dataset",
            "Source file",
            "Processed file",
            "Processed rows",
            "Rows loaded in notebook",
            "Columns in processed file",
            "Platform coverage",
            "Date columns",
        ],
        "Value": [
            metadata["title"],
            metadata["source_file"],
            metadata["processed_file"],
            metadata["processed_rows"],
            len(df),
            metadata["column_count"],
            ", ".join(metadata["platforms"]) or "Unknown",
            ", ".join(metadata["date_columns"]) or "None",
        ],
    }
)


In [ ]:
quality = (
    pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_ratio": df.isna().mean().round(4),
            "unique_values": df.nunique(dropna=False),
        }
    )
    .sort_values(["missing_ratio", "unique_values"], ascending=[False, False])
)
quality.head(15)


In [ ]:
candidate_columns = [col for col in ["platform", "record_type", "region", "sentiment", "content_type", "primary_topic"] if col in df.columns]
display_frames = []
for column in candidate_columns:
    counts = df[column].fillna("Unknown").astype(str).value_counts().head(10)
    display_frames.append(pd.DataFrame({"column": column, "value": counts.index, "count": counts.values}))
pd.concat(display_frames, ignore_index=True) if display_frames else pd.DataFrame(columns=["column", "value", "count"])


In [ ]:
time_columns = [col for col in ["post_date", "published_at", "trending_date"] if col in df.columns]
if time_columns:
    time_col = time_columns[0]
    trend = (
        df.dropna(subset=[time_col])
        .assign(year_month=lambda frame: frame[time_col].dt.to_period("M").astype(str))
        .groupby("year_month", as_index=False)
        .size()
        .rename(columns={"size": "records"})
    )
    plt.figure(figsize=(12, 4))
    sns.lineplot(data=trend, x="year_month", y="records", marker="o", linewidth=2.2, color="#D96F32")
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Monthly record volume based on {time_col}")
    plt.xlabel("Month")
    plt.ylabel("Records")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("No date column is available in this processed dataset.")


In [ ]:
numeric_candidates = [col for col in ["views", "likes", "shares", "comments", "dislikes", "total_engagement", "engagement_rate", "text_length", "word_count"] if col in df.columns]
if numeric_candidates:
    summary = df[numeric_candidates].describe().T.sort_values("mean", ascending=False)
    display(summary)
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(10, 6))
        sns.heatmap(df[numeric_candidates].corr(numeric_only=True), annot=True, cmap="YlOrBr", fmt=".2f")
        plt.title("Correlation snapshot")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric features were found in this processed dataset.")


In [ ]:
if "primary_topic" in df.columns:
    topics = (
        df.loc[df["primary_topic"].fillna("no_explicit_topic") != "no_explicit_topic", "primary_topic"]
        .astype(str)
        .value_counts()
        .head(12)
        .sort_values()
    )
    if not topics.empty:
        plt.figure(figsize=(10, 6))
        topics.plot(kind="barh", color="#254D6E")
        plt.title("Most visible explicit topics")
        plt.xlabel("Records")
        plt.ylabel("Topic")
        plt.tight_layout()
        plt.show()
    else:
        print("The dataset does not expose explicit hashtags or tags often enough for a topic chart.")
